# 05 · Distillation · de una API cara a un modelo local

> Misma tesis, pero los datos los genera un MODELO: un teacher de OpenAI (gpt-5-mini) escribe 200 Q&A
> de programación, fine-tuneamos un student local con QLoRA, y el student acaba respondiendo casi como
> el teacher... corriendo en tu T4 por centimos.

## Qué aprendemos

1. **Distillation black-box** : usar un teacher por API para generar datos y entrenar un student local.
2. Construir un **dataset en HuggingFace de forma incremental** (cada run anade filas, no borra).
3. El argumento que lo justifica: **coste** (API teacher vs local student) + calidad (LLM-as-judge).

## Por qué importa

Es como bajas un modelo caro de API a uno propio y barato sin perder casi calidad. La técnica detras de
muchos modelos "pequeños que rinden": aprenden de uno grande.

## Conecta con

- **Slides de Fine-tuning · Deck 2** : Distillation (teacher -> student, soft targets / black-box).
- **Lab 03/04** : misma receta QLoRA, pero aquí los datos los crea el teacher.
- **Módulo de Evaluación** : LLM-as-judge como método de evaluación.

## Stack

`openai` (teacher `gpt-5-mini`) · `unsloth` (student `gemma-4-E2B-it` en 4-bit; `E4B` con `MODEL_SIZE="big"`) · `datasets` + `huggingface_hub`.

## Hardware y coste

- T4 (Colab Free) para el student. Ejecución de referencia (oct-2026): ~30 min en total (~10 de entrenamiento).
- **Necesita `OPENAI_API_KEY`** (los otros labs no). Generar 200 Q&A + el eval con gpt-5-mini cuesta
  unos **centimos**. Cacheamos a HF para no re-pagar.


---

## Setup


In [1]:
!pip install -qU openai unsloth datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.1 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.2 MB/s eta 0:00:00


In [2]:
import os, json, time, getpass
from openai import OpenAI

# Clave de OpenAI (no se guarda en el notebook)
os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
client = OpenAI()

TEACHER = "gpt-5-mini"   # teacher barato; sube a "gpt-5" si quieres mas calidad

---

# 1 · El teacher · bueno, pero es API de pago

Le preguntamos cosas de programación. Responde genial. Pero cada llamada cuesta y depende de la API.


In [3]:
def ask_teacher(question, max_tokens=800):
    r = client.chat.completions.create(
        model=TEACHER,
        messages=[
            {"role": "system", "content": "Eres un mentor de programacion en Python. Responde claro y conciso, con un snippet corto cuando ayude."},
            {"role": "user", "content": question},
        ],
        max_completion_tokens=max_tokens,
        reasoning_effort="minimal",   # gpt-5-mini es de razonamiento: sin esto los tokens se van en "pensar" y el texto sale vacio
    )
    return r.choices[0].message.content.strip()

for q in ["¿Que es un decorador en Python y para que sirve?",
          "¿Diferencia entre lista y tupla?"]:
    print("P:", q)
    print("R:", ask_teacher(q), "\n")

P: ¿Que es un decorador en Python y para que sirve?


R: Un decorador en Python es una función (o callable) que recibe otra función (o método) y devuelve una función nueva o modificada. Se usa para añadir o cambiar comportamiento de funciones/métodos sin modificar su código original. Se aplica con la sintaxis @decorador encima de la definición.

Para qué sirve (ejemplos comunes):
- Medir tiempo de ejecución.
- Controlar acceso / autenticación.
- Registro (logging).
- Validación de argumentos.
- Memoización (caché).
- Reintentos, sincronización (locks), etc.

Ejemplo simple: medir tiempo de ejecución

```python
import time
def tiempo(func):
    def envoltura(*args, **kwargs):
        inicio = time.time()
        resultado = func(*args, **kwargs)
        print(f"{func.__name__} tardó {time.time() - inicio:.4f} s")
        return resultado
    return envoltura

@tiempo
def suma(n):
    return sum(range(n))

suma(1000000)
```

Notas importantes:
- Usa functools.wraps para preservar metadatos (nombre, docstring).
- Decoradores también pueden a

R: Principales diferencias entre lista y tupla en Python:

- Mutabilidad:
  - Lista: mutable — puedes añadir, eliminar o cambiar elementos (append, pop, insert, remove, asignación por índice).
  - Tupla: inmutable — no puedes modificar sus elementos después de crearla.

- Sintaxis:
  - Lista: se crea con corchetes: [1, 2, 3]
  - Tupla: con paréntesis o sin ellos: (1, 2, 3) o 1, 2, 3. Para una tupla de un elemento: (x,) o x,

- Uso típico:
  - Lista: colecciones homogéneas o que cambian a lo largo del tiempo.
  - Tupla: datos heterogéneos, registros, o cuando quieres garantizar inmutabilidad (por ejemplo, como clave de diccionario si contiene solo objetos hashables).

- Rendimiento y tamaño:
  - Tuplas suelen ser ligeramente más rápidas y ocupar menos memoria que listas (por ser inmutables y optimizadas).

- Operaciones y métodos:
  - Lista: muchos métodos (append, extend, sort, reverse, pop, remove, clear, etc.).
  - Tupla: pocos métodos (count, index).

- Hashabilidad:
  - Lista: no e

---

# 2 · Generamos el dataset · y lo subimos a HF (incremental)

Primero hacemos **login en HuggingFace** y eliges el nombre de TU dataset. Cada vez que ejecutes la
generación se **anaden** filas al dataset del Hub (no se borra nada).


In [5]:
from huggingface_hub import login, whoami
login(token=getpass.getpass("HF token (write): "))
HF_USER = whoami()["name"]

# El alumno pone su nombre de usuario / repo aqui
DATASET_REPO = f"{HF_USER}/programming-distill-qa"
print("Dataset destino:", DATASET_REPO)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Dataset destino: ericrisco/programming-distill-qa


In [6]:
# El teacher genera pares Q&A en lotes (mas barato que uno a uno)
def gen_batch(k=20):
    prompt = (
        f"Genera {k} pares pregunta-respuesta para un mentor de programacion en Python. "
        "Preguntas variadas: conceptos, buenas practicas, snippets cortos, errores comunes. "
        "Responde SOLO un JSON array de objetos con las claves \"question\" y \"answer\"."
    )
    r = client.chat.completions.create(
        model=TEACHER,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=3000,
        reasoning_effort="minimal",
    )
    txt = r.choices[0].message.content.strip()
    txt = txt.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(txt)

N_TARGET = 200
new_rows = []
while len(new_rows) < N_TARGET:
    try:
        new_rows.extend(gen_batch(20))
        print(f"  generados {len(new_rows)}/{N_TARGET}")
    except Exception as e:
        print("  lote fallido, reintento:", str(e)[:80])
new_rows = new_rows[:N_TARGET]
print("Total nuevos:", len(new_rows))

  generados 20/200


  generados 39/200


  generados 58/200


  generados 77/200


  generados 97/200


...

  generados 137/200


  generados 157/200


  generados 177/200


  generados 196/200


  generados 216/200
Total nuevos: 200


In [7]:
from datasets import Dataset, load_dataset, concatenate_datasets

new_ds = Dataset.from_list(new_rows)

# Cargar lo que ya hubiera en el Hub y ACUMULAR (nunca borrar)
try:
    existing = load_dataset(DATASET_REPO, split="train")
    combined = concatenate_datasets([existing, new_ds])
    print(f"existian {len(existing)} + {len(new_ds)} nuevas = {len(combined)} filas")
except Exception:
    combined = new_ds
    print(f"dataset nuevo: {len(combined)} filas")

combined.push_to_hub(DATASET_REPO)
print("subido a:", DATASET_REPO)

README.md:   0%|          | 0.00/306 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 36.1kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

existian 200 + 200 nuevas = 400 filas


Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

subido a: ericrisco/programming-distill-qa


---

# 3 · El student local · el "antes"

Cargamos un modelo pequeño con Unsloth. Elige tamaño según tu GPU.


In [8]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

MODEL_SIZE = "small"   # "small" = E2B (T4 free) · "big" = E4B
MODELS = {"small": "unsloth/gemma-4-E2B-it", "big": "unsloth/gemma-4-E4B-it"}

model, tokenizer = FastModel.from_pretrained(
    model_name = MODELS[MODEL_SIZE],
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.10.1: Fast Gemma4 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [12]:
@torch.no_grad()
def ask_student(question, max_new_tokens=1000):
    try:
      msgs = [{"role": "user", "content": [{"type": "text", "text": question}]}]
      # Tokenizamos en un paso: vale para tokenizer de texto y para el processor multimodal de Gemma 4
      inputs = tokenizer.apply_chat_template(
          msgs, add_generation_prompt=True, tokenize=True,
          return_tensors="pt", return_dict=True,
      ).to("cuda")
      out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
      return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    except Exception as e:
        print("  pregunta fallida", str(e)[:80])

# Preguntas de test (held-out, no necesariamente en el set generado)
TEST_Q = [
    "¿Que hace yield en una funcion?",
    "¿Como capturo excepciones especificas en Python?",
    "¿Que es una list comprehension? Da un ejemplo.",
    "¿Para que sirve __init__ en una clase?",
    "¿Diferencia entre == e is?",
]

def judge(question, answer):
    prompt = (f"Pregunta de programacion: {question}\nRespuesta del alumno: {answer}\n"
              "Puntua la respuesta de 1 a 5 (correcta, clara, util). Responde SOLO el numero.")
    r = client.chat.completions.create(model=TEACHER,
        messages=[{"role": "user", "content": prompt}], max_completion_tokens=1000, reasoning_effort="minimal")
    try:
        return int("".join(ch for ch in r.choices[0].message.content if ch.isdigit())[:1])
    except Exception:
        return 0

import time

def evaluate_student():
    """Calidad (juez 1-5), longitud media (palabras) y segundos por respuesta del student."""
    t0 = time.time()
    answers = [ask_student(q) or "" for q in TEST_Q]
    secs = (time.time() - t0) / len(TEST_Q)
    score = sum(judge(q, a) for q, a in zip(TEST_Q, answers)) / len(TEST_Q)
    words = sum(len(a.split()) for a in answers) / len(answers)
    return score, words, secs

# Referencia: cómo de largas son las respuestas del teacher en el dataset que ha generado
words_teacher = sum(len(r["answer"].split()) for r in new_rows) / len(new_rows)

score_base, words_base, secs_base = evaluate_student()
print(f"Student BASE · nota (juez gpt-5-mini): {score_base:.2f} / 5 · {words_base:.0f} palabras/resp · {secs_base:.0f} s/resp")
print(f"Teacher (dataset) · {words_teacher:.0f} palabras/resp")

Student BASE · nota (juez gpt-5-mini): 5.00 / 5 · 522 palabras/resp · 143 s/resp
Teacher (dataset) · 36 palabras/resp


---

# 4 · QLoRA · destilamos el teacher en el student

Entrenamos con el dataset acumulado (el que esta en el Hub).


In [13]:
model = FastModel.get_peft_model(
    model,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    finetune_vision_layers = False,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    random_state = 3407,
)

# El dataset combinado (question, answer) -> conversaciones -> texto
def to_text(ex):
    convs = [
        {"role": "user",      "content": [{"type": "text", "text": ex["question"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": ex["answer"]}]},
    ]
    return {"text": tokenizer.apply_chat_template(convs, tokenize=False)}

train_data = combined.map(to_text, remove_columns=combined.column_names)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [17]:
print(train_data[0]["text"])

<bos><|turn>user
¿Cuál es la diferencia entre listas y tuplas en Python y cuándo usar cada una?<turn|>
<|turn>model
Las listas son mutables (puedes modificar, añadir o eliminar elementos) y se crean con corchetes: [1,2,3]. Las tuplas son inmutables (no puedes cambiar su contenido) y se crean con paréntesis: (1,2,3). Usa listas cuando necesites modificar la colección, y tuplas cuando quieras asegurar inmutabilidad, usar como claves en diccionarios (siempre y cuando sus elementos sean hashables) o cuando quieras transmitir que los datos no deben cambiar.<turn|>



In [18]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = train_data,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)
trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 3 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,158,208 of 5,128,455,712 (0.47% trained)


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss
10,3.814562
20,1.743229
30,1.369780
40,1.210743
50,1.096753
60,0.899837
70,0.848986
80,0.803775
90,0.823427
100,0.800655


TrainOutput(global_step=150, training_loss=1.0858154773712159, metrics={'train_runtime': 541.0325, 'train_samples_per_second': 2.218, 'train_steps_per_second': 0.277, 'total_flos': 1635332662267392.0, 'train_loss': 1.0858154773712159, 'epoch': 3.0})

---

# 5 · El "después" · el student destilado

Qué mirar: en preguntas genéricas de Python el Gemma 4 E2B base ya es bueno, así que la nota del juez
(1-5) suele salir alta antes y después (el juez satura). Lo que la distillation transfiere aquí es el
**estilo del teacher**: respuestas igual de correctas pero mucho más cortas y directas, como las del
dataset. Menos tokens generados = respuestas más rápidas y más baratas de servir. Para ver subir la
**calidad** necesitas un dominio donde el student base falle (tu API interna, tu normativa, tu jerga):
la receta es la misma.


In [19]:
print("=== Ejemplos del student destilado ===\n")
for q in TEST_Q[:2]:
    print("P:", q)
    print("R:", ask_student(q), "\n")

score_ft, words_ft, secs_ft = evaluate_student()
print(f"                 nota juez   palabras/resp   s/resp")
print(f"  Student base     {score_base:4.2f}        {words_base:5.0f}        {secs_base:5.0f}")
print(f"  Destilado        {score_ft:4.2f}        {words_ft:5.0f}        {secs_ft:5.0f}")
print(f"  Teacher (datos)    -          {words_teacher:5.0f}")

=== Ejemplos del student destilado ===

P: ¿Que hace yield en una funcion?


R: yield produce un iterador perezoso que devuelve valores uno a uno y mantiene estado entre llamadas. Permite generar secuencias grandes sin cargar todo en memoria. Ejemplo: def contador(n): for i in range(n): yield i. 

P: ¿Como capturo excepciones especificas en Python?


R: Usa try/except con la excepción específica: try: ... except ValueError: ... except TypeError: ... finally: ... Evita usar except: sin especificar, captura todas las excepciones y oculta errores. 



                 nota juez   palabras/resp   s/resp
  Student base     5.00          522          143
  Destilado        5.00           30           10
  Teacher (datos)    -             36


---

# 6 · El número que importa · coste

El student destilado responde con la calidad y el estilo del teacher, pero corre local. Coste aproximado por 1000 consultas:


In [20]:
# Precio de lista de gpt-5-mini (oct-2026, compruébalo en openai.com/api/pricing):
# 0,25 USD por 1M tokens de entrada y 2,00 USD por 1M de salida.
# Asumimos ~100 tokens de pregunta (+ system) y ~500 de respuesta por consulta.
IN_TOK, OUT_TOK = 100, 500
PRICE_TEACHER_PER_1K_QUERIES = 1000 * (IN_TOK * 0.25 + OUT_TOK * 2.00) / 1e6   # ~1,03 USD via API
PRICE_LOCAL_PER_1K_QUERIES   = 0.02   # USD aprox (electricidad/GPU amortizada)

print("Coste por 1000 consultas (ilustrativo):")
print(f"  Teacher (API {TEACHER}): ~${PRICE_TEACHER_PER_1K_QUERIES:.2f}")
print(f"  Student local (destilado): ~${PRICE_LOCAL_PER_1K_QUERIES:.2f}")
print(f"  Ahorro: ~{(1 - PRICE_LOCAL_PER_1K_QUERIES/PRICE_TEACHER_PER_1K_QUERIES)*100:.0f}%")
print("\nNota: precios ilustrativos · el punto es el ORDEN DE MAGNITUD, no el numero exacto.")

Coste por 1000 consultas (ilustrativo):
  Teacher (API gpt-5-mini): ~$1.02
  Student local (destilado): ~$0.02
  Ahorro: ~98%

Nota: precios ilustrativos · el punto es el ORDEN DE MAGNITUD, no el numero exacto.


---

# Cierre · lo que nos llevamos

- **Distillation black-box**: un teacher de API genero los datos y un student local aprendio de ellos.
- Construimos un **dataset en HF de forma incremental** (cada run anade, no borra) que puedes seguir
  ampliando entre clases.
- El student destilado mantiene la calidad (según el juez) y copia el estilo conciso del teacher, **corriendo
  local por una fracción del coste**. En preguntas genéricas el juez 1-5 satura: para ver subir la calidad,
  destila en un dominio donde el student base falle.

## Hacia el alignment

Tu modelo ya sabe tareas, estilos y dominios. Pero, ¿se comporta como debe? El siguiente bloque es
alignment: Reward Model + RLHF y luego DPO/ORPO.


---

## Guarda tu adapter (para el Lab 09)

Sube el adapter de programación destilada para poder evaluarlo (antes/después) en el capstone de evaluación.


In [21]:
ADAPTER_REPO = f"{HF_USER}/prog-distill-adapter"
model.push_to_hub(ADAPTER_REPO)       # sube solo el adapter (pesa poco)
tokenizer.push_to_hub(ADAPTER_REPO)
print("adapter guardado en", ADAPTER_REPO)

Unsloth: Restored added_tokens_decoder metadata in ericrisco/prog-distill-adapter/tokenizer_config.json.


adapter guardado en ericrisco/prog-distill-adapter
